# Titanic: previsão de sobrevivência

O case mais conhecido do Kaggle, usado aqui não pela originalidade do tema,
mas porque é um bom terreno para mostrar um pipeline de classificação
completo e correto: EDA orientada a hipótese, feature engineering,
pré-processamento com `ColumnTransformer`, comparação de modelos com
validação cruzada e avaliação com múltiplas métricas.

**Pergunta de negócio (adaptada):** dado o perfil de um passageiro (classe,
sexo, idade, tarifa paga, composição familiar a bordo), qual a
probabilidade de ele ter sobrevivido ao naufrágio?

Dataset: `data/titanic.csv` (891 passageiros do Titanic, dataset público
clássico do Kaggle - *Titanic: Machine Learning from Disaster*).

## Importando bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, RocCurveDisplay

sns.set_style("whitegrid")
RANDOM_STATE = 42

## Carregando os dados

In [ ]:
df = pd.read_csv("data/titanic.csv")
print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.isnull().sum().sort_values(ascending=False)

## Análise exploratória

Antes de sair criando variáveis, vale confirmar no próprio dado os padrões
que já são conhecidos historicamente sobre o naufrágio: prioridade de
mulheres e crianças nos botes salva-vidas, e melhor acesso a botes para
passageiros de 1ª classe.

In [ ]:
df.groupby("Sex")["Survived"].mean().sort_values(ascending=False)

In [ ]:
df.groupby("Pclass")["Survived"].mean().sort_values(ascending=False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

sns.barplot(data=df, x="Pclass", y="Survived", hue="Sex", ax=axes[0])
axes[0].set_title("Taxa de sobrevivência por classe e sexo")

sns.histplot(data=df, x="Age", hue="Survived", multiple="stack", bins=30, ax=axes[1])
axes[1].set_title("Distribuição de idade por sobrevivência")

plt.tight_layout()
plt.show()

## Feature engineering

Duas transformações que costumam ajudar nesse dataset e que não são óbvias
só olhando as colunas originais:

- **Título extraído do nome** (`Mr`, `Mrs`, `Miss`, `Master`...) carrega
  informação de idade/status social mesmo quando `Age` está faltando.
- **Tamanho da família a bordo** (`SibSp + Parch + 1`) e um indicador de
  "viajando sozinho" - famílias muito grandes ou passageiros sozinhos
  tendem a ter taxas de sobrevivência diferentes da média.

In [ ]:
df["Title"] = df["Name"].str.extract(r",\s*([^.]*)\.")
# agrupa títulos raros para não criar categorias com 1-2 observações
common_titles = {"Mr", "Miss", "Mrs", "Master"}
df["Title"] = df["Title"].apply(lambda t: t if t in common_titles else "Rare")

df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

df[["Title", "FamilySize", "IsAlone"]].head()

## Pipeline de pré-processamento e modelagem

In [ ]:
numeric_features = ["Age", "Fare", "FamilySize"]
categorical_features = ["Pclass", "Sex", "Embarked", "Title", "IsAlone"]

X = df[numeric_features + categorical_features]
y = df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), categorical_features),
])

In [ ]:
models = {
    "logistic_regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "random_forest": RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE),
}

cv_results = {}
for name, model in models.items():
    pipe = Pipeline([("preprocess", preprocess), ("model", model)])
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring="accuracy")
    cv_results[name] = scores
    print(f"{name}: acurácia média (5-fold CV) = {scores.mean():.3f} +/- {scores.std():.3f}")

## Ajuste fino e avaliação do modelo final

Random Forest costuma performar melhor nesse dataset por capturar
interações não-lineares (ex: "classe alta E mulher" importa mais que os
dois fatores somados linearmente). Um `GridSearchCV` pequeno para ajustar
profundidade e número de árvores, e então avaliação no conjunto de teste
que ficou de fora de todo o processo de tuning.

In [ ]:
param_grid = {
    "model__n_estimators": [200, 400],
    "model__max_depth": [4, 6, 8, None],
    "model__min_samples_leaf": [1, 3, 5],
}

rf_pipe = Pipeline([("preprocess", preprocess), ("model", RandomForestClassifier(random_state=RANDOM_STATE))])

grid = GridSearchCV(rf_pipe, param_grid, cv=5, scoring="roc_auc", n_jobs=-1)
grid.fit(X_train, y_train)

print("Melhores parâmetros:", grid.best_params_)
print(f"Melhor AUC (CV): {grid.best_score_:.3f}")

In [ ]:
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=["Não sobreviveu", "Sobreviveu"]))
print(f"AUC no teste: {roc_auc_score(y_test, y_proba):.3f}")

In [ ]:
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Previsto: não", "Previsto: sim"],
            yticklabels=["Real: não", "Real: sim"])
plt.title("Matriz de confusão - conjunto de teste")
plt.show()

In [ ]:
RocCurveDisplay.from_estimator(best_model, X_test, y_test)
plt.title("Curva ROC - conjunto de teste")
plt.show()

## O que fica desse projeto

O ponto principal aqui não é bater recorde de acurácia num dataset de 891
linhas - é o pipeline: `ColumnTransformer` separando tratamento de
numéricas e categóricas, `Pipeline` amarrando pré-processamento e modelo
num objeto só (evita vazamento de dados entre treino/teste), validação
cruzada para comparar modelos antes de escolher, `GridSearchCV` para
ajuste de hiperparâmetros, e avaliação com métricas que fazem sentido para
classificação binária (não só acurácia, mas matriz de confusão e AUC).

Esse é o mesmo esqueleto que uso em problemas reais de classificação -
churn, fraude, propensão de compra - trocando o dataset e as features.